# Домашнее задание 2.3: Связанные колебания и жёсткость

Динамика системы двух связанных масс описывается уравнениями:
$$ m \ddot{x}_1 = -k_0 x_1 - k_c(x_1 - x_2) - c(\dot{x}_1 - \dot{x}_2) $$
$$ m \ddot{x}_2 = -k_0 x_2 - k_c(x_2 - x_1) - c(\dot{x}_2 - \dot{x}_1) $$

Система сводится к векторному виду $\dot{\mathbf{u}} = \mathbf{A}\mathbf{u}$, где $\mathbf{u} = [x_1, x_2, v_1, v_2]^T$. Это позволяет использовать матричные операции для RK4 и получить аналитическое выражение шага для неявного метода трапеций:
$$ \mathbf{u}_{n+1} = \left(\mathbf{I} - \frac{\Delta t}{2}\mathbf{A}\right)^{-1} \left(\mathbf{I} + \frac{\Delta t}{2}\mathbf{A}\right) \mathbf{u}_n $$

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.animation as animation
from IPython.display import HTML
from scipy.signal import find_peaks

# Увеличиваем лимит памяти для анимации до 100 МБ
plt.rcParams['animation.embed_limit'] = 100.0

# Глобальная матрица системы
def get_A_matrix(m, k0, kc, c):
    return np.array([
        [0, 0, 1, 0],
        [0, 0, 0, 1],
        [-(k0+kc)/m, kc/m, -c/m, c/m],
        [kc/m, -(k0+kc)/m, c/m, -c/m]
    ])

def rk4_step(u, dt, A):
    k1 = A @ u
    k2 = A @ (u + 0.5 * dt * k1)
    k3 = A @ (u + 0.5 * dt * k2)
    k4 = A @ (u + dt * k3)
    return u + (dt / 6.0) * (k1 + 2*k2 + 2*k3 + k4)

def verlet_step(x, v, a, dt, m, k0, kc):
    x_next = x + v * dt + 0.5 * a * dt**2
    a_next = np.array([
        (-k0*x_next[0] - kc*(x_next[0] - x_next[1])) / m,
        (-k0*x_next[1] - kc*(x_next[1] - x_next[0])) / m
    ])
    v_next = v + 0.5 * (a + a_next) * dt
    return x_next, v_next, a_next

def implicit_trapezoidal_step(u, dt, A):
    I = np.eye(4)
    M_inv = np.linalg.inv(I - 0.5 * dt * A)
    M_fwd = I + 0.5 * dt * A
    return (M_inv @ M_fwd) @ u

# Генерация координат пружины для анимации (зигзаг)
def get_spring(x_start, x_end, n_coils=10, width=0.3):
    length = x_end - x_start
    x = np.linspace(x_start, x_end, n_coils)
    y = np.zeros(n_coils)
    y[1:-1:2] = width
    y[2:-1:2] = -width
    return x, y

## 🟢 Базовый уровень: Слабая связь и биения

Слабая связь ($k_c \ll k_0$) без демпфирования ($c=0$). 
Аналитические частоты: $\omega_1 = \sqrt{k_0/m}$ (синфазная) и $\omega_2 = \sqrt{(k_0+2k_c)/m}$ (противофазная).
Период биений: $T_{beat} = 2\pi / (\omega_2 - \omega_1)$.

In [ ]:

from scipy.signal import hilbert

m, k0, kc, c = 1.0, 10.0, 0.5, 0.0
A_weak = get_A_matrix(m, k0, kc, c)

# Аналитические собственные частоты
w1 = np.sqrt(k0 / m)
w2 = np.sqrt((k0 + 2 * kc) / m)
T_beat_analytic = 2 * np.pi / (w2 - w1)


def simulate_weak(dt, t_max=80.0):
    n_steps = int(round(t_max / dt))
    t_arr = np.arange(n_steps + 1) * dt

    u_rk4 = np.zeros((n_steps + 1, 4))
    u_ver = np.zeros_like(u_rk4)

    u_rk4[0] = u_ver[0] = [1.0, 0.0, 0.0, 0.0]

    x_v = u_ver[0, :2].copy()
    v_v = u_ver[0, 2:].copy()
    a_v = (-(k0 + kc)*x_v + kc*x_v[::-1]) / m

    for i in range(n_steps):
        u_rk4[i+1] = rk4_step(u_rk4[i], dt, A_weak)

        x_v, v_v, a_v = verlet_step(
            x_v, v_v, a_v, dt, m, k0, kc
        )
        u_ver[i+1] = np.r_[x_v, v_v]

    return t_arr, u_rk4, u_ver


# Точное решение при x1(0)=1, x2(0)=0, v1=v2=0
def weak_exact(t):
    s = 0.5 * np.cos(w1 * t)
    d = 0.5 * np.cos(w2 * t)

    vs = -0.5 * w1 * np.sin(w1 * t)
    vd = -0.5 * w2 * np.sin(w2 * t)

    return np.column_stack((
        s + d, s - d, vs + vd, vs - vd
    ))


def weak_total_energy(U):
    return (
        0.5*m*(U[:, 2]**2 + U[:, 3]**2)
        + 0.5*k0*(U[:, 0]**2 + U[:, 1]**2)
        + 0.5*kc*(U[:, 0] - U[:, 1])**2
    )


# Одинаковые шаги для двух методов
h_weak = [0.20, 0.10, 0.05, 0.02, 0.01]

weak_runs = {}
err_rk4 = []
err_verlet = []

print(
    f"{'h':>7} {'Ошибка RK4':>15} "
    f"{'Ошибка Верле':>15} "
    f"{'Дрейф E RK4':>15} "
    f"{'Дрейф E Верле':>15}"
)

for h in h_weak:
    tt, ur, uv = simulate_weak(h)
    weak_runs[h] = (tt, ur, uv)

    exact = weak_exact(tt)

    er = np.max(
        np.linalg.norm(
            ur[:, :2] - exact[:, :2], axis=1
        )
    )
    ev = np.max(
        np.linalg.norm(
            uv[:, :2] - exact[:, :2], axis=1
        )
    )

    err_rk4.append(er)
    err_verlet.append(ev)

    E_r = weak_total_energy(ur)
    E_v = weak_total_energy(uv)

    dr = np.max(np.abs(E_r/E_r[0] - 1))
    dv = np.max(np.abs(E_v/E_v[0] - 1))

    print(
        f"{h:7.3f} {er:15.6e} {ev:15.6e} "
        f"{dr:15.6e} {dv:15.6e}"
    )


# Прежние переменные сохраняем для анимации
dt1, dt2 = 0.01, 0.10

t1, rk4_1, ver_1 = weak_runs[dt1]
t2, rk4_2, ver_2 = weak_runs[dt2]


# Численное определение периода биений
# Продлеваем интегрирование для нескольких биений
h_env = 0.02
t_long, rk_long, _ = simulate_weak(h_env, t_max=240.0)

# Огибающая сигнала x1(t)
envelope = np.abs(hilbert(rk_long[:, 0]))

minima, _ = find_peaks(
    -envelope,
    distance=int(25/h_env),
    prominence=0.5
)

# Исключаем границы интервала:
# там преобразование Гильберта менее точно
minima = minima[
    (t_long[minima] > 10)
    & (t_long[minima] < 230)
]

T_beat_num = np.mean(np.diff(t_long[minima]))


# Частоты собственных мод из численного решения
s_num = 0.5*(rk_long[:, 0] + rk_long[:, 1])
d_num = 0.5*(rk_long[:, 0] - rk_long[:, 1])

def frequency_from_peaks(signal, tt):
    peaks, _ = find_peaks(signal)
    peaks = peaks[
        (tt[peaks] > 5)
        & (tt[peaks] < tt[-1] - 5)
    ]
    return 2*np.pi / np.mean(np.diff(tt[peaks]))


w1_num = frequency_from_peaks(s_num, t_long)
w2_num = frequency_from_peaks(d_num, t_long)

print(
    f"\nСинфазная частота: "
    f"аналит. {w1:.6f}, числ. {w1_num:.6f} рад/с"
)
print(
    f"Противофазная частота: "
    f"аналит. {w2:.6f}, числ. {w2_num:.6f} рад/с"
)
print(
    f"Период биений: "
    f"аналит. {T_beat_analytic:.4f}, "
    f"числ. {T_beat_num:.4f} с"
)


# Кинетические и полная механическая энергия
Ek1 = 0.5*m*ver_1[:, 2]**2
Ek2 = 0.5*m*ver_1[:, 3]**2
E_total = weak_total_energy(ver_1)

fig, axes = plt.subplots(2, 2, figsize=(14, 9))

axes[0, 0].plot(
    t1, rk4_1[:, 0], label="x1, RK4", lw=0.8
)
axes[0, 0].plot(
    t1, rk4_1[:, 1], label="x2, RK4", lw=0.8
)
axes[0, 0].set_title("Биения при слабой связи, h=0.01")
axes[0, 0].set_xlabel("t, с")
axes[0, 0].set_ylabel("Смещение")
axes[0, 0].legend()

axes[0, 1].plot(
    t1, Ek1, label="Кинетическая энергия массы 1", lw=0.7
)
axes[0, 1].plot(
    t1, Ek2, label="Кинетическая энергия массы 2", lw=0.7
)
axes[0, 1].plot(
    t1, E_total, "k", label="Полная энергия", lw=1.6
)
axes[0, 1].set_title("Энергии (Верле, h=0.01)")
axes[0, 1].set_xlabel("t, с")
axes[0, 1].set_ylabel("Энергия")
axes[0, 1].legend()

axes[1, 0].loglog(
    h_weak, err_rk4, "o-", label="RK4"
)
axes[1, 0].loglog(
    h_weak, err_verlet, "s-", label="Верле"
)
axes[1, 0].set_title("Максимальная ошибка смещений")
axes[1, 0].set_xlabel("Шаг h")
axes[1, 0].set_ylabel("Ошибка за 80 с")
axes[1, 0].legend()

axes[1, 1].plot(
    t_long, envelope, label="Огибающая |Hilbert(x1)|"
)
axes[1, 1].plot(
    t_long[minima],
    envelope[minima],
    "ro",
    label="Минимумы огибающей"
)
axes[1, 1].set_xlim(0, 230)
axes[1, 1].set_title(
    f"Численный период биений ≈ {T_beat_num:.2f} с"
)
axes[1, 1].set_xlabel("t, с")
axes[1, 1].set_ylabel("Амплитуда")
axes[1, 1].legend()

for ax in axes.flat:
    ax.grid(alpha=0.3)

plt.tight_layout()
plt.show()



## 🟡 Продвинутый уровень: Жёсткость и неявные методы

Увеличим жёсткость соединительной пружины ($k_c\gg k_0$) и добавим демпфер ($c>0$).

Введём новые координаты:

$$
q=\frac{x_1+x_2}{2},\qquad d=x_1-x_2.
$$

Исходная система распадается на два независимых уравнения:

$$
\ddot q+\frac{k_0}{m}q=0,
$$

$$
\ddot d+\frac{2c}{m}\dot d+
\frac{k_0+2k_c}{m}d=0.
$$

Первая мода описывает медленные совместные колебания масс, вторая — быстрое относительное движение.

Выберем параметры:

$$
m=1,\quad k_0=2,\quad k_c=400,\quad c=10.
$$

Медленная мода имеет частоту

$$
\omega_s=\sqrt{2}\approx1.414.
$$

Собственные значения быстрой моды:

$$
\lambda_{\mathrm{fast}}=-10\pm26.495i.
$$

Следовательно, относительное движение затухает с характерным временем

$$
\tau=\frac{m}{c}=0.1.
$$

### Устойчивость RK4

Функция устойчивости явного метода RK4:

$$
R_{\mathrm{RK4}}(z)=
1+z+\frac{z^2}{2}+\frac{z^3}{6}+\frac{z^4}{24}.
$$

Для устойчивости необходимо, чтобы для всех собственных значений матрицы системы выполнялось

$$
|R_{\mathrm{RK4}}(h\lambda_j)|\le1.
$$

При нарушении этого условия соответствующая численная мода растёт, даже если точное решение затухает.

### Неявный метод трапеций

Для неявного метода трапеций (Адамса–Моултона второго порядка) функция устойчивости имеет вид

$$
R_{\mathrm{trap}}(z)=\frac{1+z/2}{1-z/2}.
$$

Этот метод A-устойчив: для $\operatorname{Re}z\le0$ выполняется $|R_{\mathrm{trap}}(z)|\le1$.

Однако метод не является L-устойчивым: при очень больших шагах быстро затухающие компоненты могут подавляться недостаточно эффективно.

### Основной вопрос

Можно ли после затухания быстрого относительного движения увеличить шаг интегрирования?

Проверим это экспериментально, сравнивая RK4 и неявный метод трапеций как с точки зрения устойчивости, так и с точки зрения точности медленной моды.


In [ ]:

# Более жёсткая связь
k0_s, kc_s, c_s = 2.0, 400.0, 10.0

A_stiff = get_A_matrix(m, k0_s, kc_s, c_s)

u0_stiff = np.array([1.0, 0.0, 0.0, 0.0])

# Характеристики медленной и быстрой мод
omega_s = np.sqrt(k0_s / m)
gamma = c_s / m
omega_fast = np.sqrt((k0_s + 2*kc_s) / m)
omega_d = np.sqrt(omega_fast**2 - gamma**2)

print(
    f"omega_s = {omega_s:.4f}, "
    f"omega_fast = {omega_fast:.4f}, "
    f"gamma = {gamma:.4f}"
)
print(f"Время затухания ≈ {1/gamma:.3f} с")


def simulate_stiff(dt, t_max=20.0):
    n_steps = int(np.floor(t_max/dt + 1e-10))
    t_arr = np.arange(n_steps + 1)*dt

    u_rk = np.empty((n_steps + 1, 4))
    u_imp = np.empty_like(u_rk)

    u_rk[0] = u_imp[0] = u0_stiff

    # Матрица неявного шага вычисляется один раз
    I = np.eye(4)

    trapezoid_matrix = np.linalg.solve(
        I - 0.5*dt*A_stiff,
        I + 0.5*dt*A_stiff
    )

    for i in range(n_steps):
        u_rk[i+1] = rk4_step(
            u_rk[i], dt, A_stiff
        )
        u_imp[i+1] = trapezoid_matrix @ u_imp[i]

    return t_arr, u_rk, u_imp


# Точное решение через нормальные координаты
def stiff_exact(t):
    q0 = (u0_stiff[0] + u0_stiff[1])/2
    d0 = u0_stiff[0] - u0_stiff[1]

    q = q0*np.cos(omega_s*t)

    d = d0*np.exp(-gamma*t)*(
        np.cos(omega_d*t)
        + (gamma/omega_d)*np.sin(omega_d*t)
    )

    return q + d/2, q - d/2, q, d


# Функции устойчивости
def R_rk4(z):
    return 1 + z + z**2/2 + z**3/6 + z**4/24


def R_trap(z):
    return (1 + z/2)/(1 - z/2)


fast_lambda = -gamma + 1j*omega_d

h_stiff = [0.01, 0.04, 0.08, 0.10, 0.12]
stiff_runs = {}

print(
    f"\n{'h':>6} {'|R_RK4|':>12} "
    f"{'|R_trap|':>12} "
    f"{'max |x1| RK4':>17} "
    f"{'Ошибка q, trap':>16}"
)

for h in h_stiff:
    tt, ur, ui = simulate_stiff(h)
    stiff_runs[h] = (tt, ur, ui)

    _, _, q_exact, _ = stiff_exact(tt)
    q_trap = 0.5*(ui[:, 0] + ui[:, 1])

    print(
        f"{h:6.2f} "
        f"{abs(R_rk4(h*fast_lambda)):12.5f} "
        f"{abs(R_trap(h*fast_lambda)):12.5f} "
        f"{np.max(np.abs(ur[:, 0])):17.4e} "
        f"{np.max(np.abs(q_trap-q_exact)):16.4e}"
    )


# Сохраняем переменные для существующей анимации
t_fine, rk_fine, imp_fine = stiff_runs[0.01]
t_coarse, rk_coarse, imp_coarse = stiff_runs[0.12]


fig, axes = plt.subplots(2, 2, figsize=(14, 9))

# 1. Начальный переходный процесс
x1_ex, x2_ex, q_ex, d_ex = stiff_exact(t_fine)

mask = t_fine <= 0.8

axes[0, 0].plot(
    t_fine[mask],
    d_ex[mask],
    "k",
    label="Точное x1-x2"
)
axes[0, 0].plot(
    t_fine[mask],
    (rk_fine[:, 0]-rk_fine[:, 1])[mask],
    "--",
    label="RK4, h=0.01"
)
axes[0, 0].plot(
    t_fine[mask],
    (imp_fine[:, 0]-imp_fine[:, 1])[mask],
    ":",
    label="Трапеции, h=0.01"
)

t08, r08, i08 = stiff_runs[0.08]
mask08 = t08 <= 0.8

axes[0, 0].plot(
    t08[mask08],
    (i08[:, 0]-i08[:, 1])[mask08],
    "o-",
    ms=3,
    label="Трапеции, h=0.08"
)

axes[0, 0].set_title("Затухание относительного движения")
axes[0, 0].set_xlabel("t, с")
axes[0, 0].set_ylabel("x1-x2")
axes[0, 0].legend(fontsize=8)


# 2. Последующее медленное движение
for h in [0.01, 0.08, 0.12]:
    tt, ur, ui = stiff_runs[h]
    late = tt >= 3

    axes[0, 1].plot(
        tt[late],
        ui[late, 0],
        label=f"Трапеции, h={h}"
    )

late = t_fine >= 3

axes[0, 1].plot(
    t_fine[late],
    x1_ex[late],
    "k--",
    label="Точное решение"
)

axes[0, 1].set_title("Медленные колебания (t >= 3 с)")
axes[0, 1].set_xlabel("t, с")
axes[0, 1].set_ylabel("x1")
axes[0, 1].legend(fontsize=8)


# 3. Устойчивость быстрой моды
# Норма относительной моды исключает
# нули, возникающие в отдельных координатах
for h in [0.08, 0.10, 0.12]:
    tt, ur, ui = stiff_runs[h]

    rel_norm = np.hypot(
        ur[:, 0]-ur[:, 1],
        (ur[:, 2]-ur[:, 3])/omega_fast
    )

    axes[1, 0].semilogy(
        tt,
        np.maximum(rel_norm, 1e-16),
        label=f"RK4, h={h}"
    )

for h in [0.08, 0.12]:
    tt, ur, ui = stiff_runs[h]

    rel_norm = np.hypot(
        ui[:, 0]-ui[:, 1],
        (ui[:, 2]-ui[:, 3])/omega_fast
    )

    axes[1, 0].semilogy(
        tt,
        np.maximum(rel_norm, 1e-16),
        "--",
        label=f"Трапеции, h={h}"
    )

axes[1, 0].set_title("Численная устойчивость быстрой моды")
axes[1, 0].set_xlabel("t, с")
axes[1, 0].set_ylabel("Норма относительной моды")
axes[1, 0].legend(fontsize=8)


# 4. Фактическое расхождение RK4
# Никакого ограничения ylim!
axes[1, 1].semilogy(
    t_coarse,
    np.maximum(np.abs(rk_coarse[:, 0]), 1e-16),
    label="|x1| RK4, h=0.12"
)

axes[1, 1].semilogy(
    t_coarse,
    np.maximum(np.abs(imp_coarse[:, 0]), 1e-16),
    label="|x1| трапеции, h=0.12"
)

axes[1, 1].semilogy(
    t_coarse,
    np.maximum(
        np.abs(stiff_exact(t_coarse)[0]), 1e-16
    ),
    "k--",
    label="|x1| точно"
)

axes[1, 1].set_title("Численный взрыв RK4")
axes[1, 1].set_xlabel("t, с")
axes[1, 1].set_ylabel("|x1| (логарифмическая шкала)")
axes[1, 1].legend(fontsize=8)


for ax in axes.flat:
    ax.grid(alpha=0.3, which="both")

plt.tight_layout()
plt.show()



### Увеличение шага после затухания быстрого процесса

Проверим возможность увеличения шага интегрирования после затухания относительного движения.

Сначала интегрируем систему с шагом $h=0.01$ до момента $t=1$.

При выбранных параметрах быстрое движение затухает с характерным временем $\tau=0.1$ с, поэтому к моменту $t=1$ его амплитуда существенно уменьшается.

Затем увеличиваем шаг до $h=0.10$ и продолжаем интегрирование до $t=20$.

Для RK4 и метода трапеций используем одинаковые моменты переключения шага.

Сравним численные решения с точным решением системы.

Особое внимание уделим медленной координате

$$
q(t)=\frac{x_1(t)+x_2(t)}{2},
$$

которая описывает совместное движение масс.


In [ ]:

# Переключение шага после быстрого переходного процесса
t_switch = 1.0
h_late = 0.10
t_end = 20.0

# В предыдущем эксперименте шаг был 0.01
index_switch = int(round(t_switch/0.01))

u_rk_start = rk_fine[index_switch].copy()
u_imp_start = imp_fine[index_switch].copy()

# Продолжаем интегрирование с новым шагом
n_late = int(round((t_end-t_switch)/h_late))
t_late = t_switch + np.arange(n_late+1)*h_late

u_rk_late = np.empty((n_late+1, 4))
u_imp_late = np.empty_like(u_rk_late)

u_rk_late[0] = u_rk_start
u_imp_late[0] = u_imp_start

I = np.eye(4)

P_late = np.linalg.solve(
    I - 0.5*h_late*A_stiff,
    I + 0.5*h_late*A_stiff
)

for i in range(n_late):
    u_rk_late[i+1] = rk4_step(
        u_rk_late[i], h_late, A_stiff
    )

    u_imp_late[i+1] = P_late @ u_imp_late[i]


# Сравнение с точным решением
x1_exact_late, _, q_exact_late, _ = stiff_exact(t_late)

q_imp_late = 0.5*(
    u_imp_late[:, 0] + u_imp_late[:, 1]
)

q_rk_late = 0.5*(
    u_rk_late[:, 0] + u_rk_late[:, 1]
)

print(
    f"Переключение шага: "
    f"h=0.01 до t={t_switch}, "
    f"затем h={h_late}"
)

print(
    "Максимальная ошибка медленной координаты q "
    "для трапеций:",
    np.max(np.abs(q_imp_late-q_exact_late))
)

print(
    "Максимальная ошибка медленной координаты q "
    "для RK4:",
    np.max(np.abs(q_rk_late-q_exact_late))
)

print(
    "Максимальная амплитуда x1 для RK4:",
    np.max(np.abs(u_rk_late[:, 0]))
)


fig, axes = plt.subplots(1, 2, figsize=(13, 4))

# Сравнение медленного движения
axes[0].plot(
    t_late,
    x1_exact_late,
    "k",
    label="Точное x1"
)
axes[0].plot(
    t_late,
    u_imp_late[:, 0],
    "--",
    label="Трапеции, h=0.10 после t=1"
)

axes[0].set_title("Медленное движение после переключения шага")
axes[0].set_xlabel("t, с")
axes[0].set_ylabel("x1")
axes[0].legend()


# Демонстрация устойчивости
axes[1].semilogy(
    t_late,
    np.maximum(np.abs(u_rk_late[:, 0]), 1e-16),
    label="RK4: |x1|"
)
axes[1].semilogy(
    t_late,
    np.maximum(np.abs(u_imp_late[:, 0]), 1e-16),
    label="Трапеции: |x1|"
)

axes[1].set_title("Устойчивость после переключения шага")
axes[1].set_xlabel("t, с")
axes[1].set_ylabel("|x1|, логарифмическая шкала")
axes[1].legend()

for ax in axes:
    ax.grid(alpha=0.3)

plt.tight_layout()
plt.show()


### Анимации систем (слабая и жесткая связь)
Отрисовка движения масс и соединительных пружин.
Для оптимизации размера встроенной анимации применяется прореживание кадров.

In [ ]:
fig_anim1, ax_a1 = plt.subplots(figsize=(10, 3))
ax_a1.set_xlim(-4, 4)
ax_a1.set_ylim(-1, 1)
ax_a1.set_title("Слабая связь: Биения (пружины отрисованы)")
ax_a1.get_yaxis().set_visible(False)

m1_w, = ax_a1.plot([], [], 'bo', ms=20, zorder=5)
m2_w, = ax_a1.plot([], [], 'ro', ms=20, zorder=5)
spr_left_w, = ax_a1.plot([], [], 'k-', lw=1.5)
spr_mid_w, = ax_a1.plot([], [], 'g-', lw=2.0)
spr_right_w, = ax_a1.plot([], [], 'k-', lw=1.5)

plt.close(fig_anim1)

# Прореживаем кадры для снижения веса анимации (отрисовка каждые 0.1 сек)
frame_step = int(0.1 / dt1) 
frames_w = len(t1) // frame_step

def update_anim1(frame):
    idx = frame * frame_step
    x1, x2 = rk4_1[idx, 0] - 1.5, rk4_1[idx, 1] + 1.5 # Базовые позиции -1.5 и 1.5
    
    m1_w.set_data([x1], [0])
    m2_w.set_data([x2], [0])
    
    xl, yl = get_spring(-4, x1)
    xm, ym = get_spring(x1, x2, n_coils=15)
    xr, yr = get_spring(x2, 4)
    
    spr_left_w.set_data(xl, yl)
    spr_mid_w.set_data(xm, ym)
    spr_right_w.set_data(xr, yr)
    
    return m1_w, m2_w, spr_left_w, spr_mid_w, spr_right_w

ani1 = animation.FuncAnimation(fig_anim1, update_anim1, frames=frames_w, interval=50, blit=True)
HTML(ani1.to_jshtml())

In [ ]:
# Анимация 2: Жёсткая связь + график относительного смещения
fig_anim2, (ax_a2, ax_graph) = plt.subplots(2, 1, figsize=(10, 6))

ax_a2.set_xlim(-4, 4)
ax_a2.set_ylim(-1, 1)
ax_a2.set_title("Сильная связь + демпфер: Синфазное движение")
ax_a2.get_yaxis().set_visible(False)

m1_s, = ax_a2.plot([], [], 'bo', ms=20, zorder=5)
m2_s, = ax_a2.plot([], [], 'ro', ms=20, zorder=5)
spr_left_s, = ax_a2.plot([], [], 'k-', lw=1.5)
spr_mid_s, = ax_a2.plot([], [], 'g-', lw=4.0) # Толстая пружина = жесткая связь
spr_right_s, = ax_a2.plot([], [], 'k-', lw=1.5)

ax_graph.set_xlim(0, 20)
ax_graph.set_ylim(-4, 4)
ax_graph.set_title("Относительное смещение (x1 - x2)")
ax_graph.set_xlabel("Время, с")
graph_rel, = ax_graph.plot([], [], 'purple', lw=2)

plt.tight_layout()
plt.close(fig_anim2)

frames_s = len(t_fine) // frame_step

def update_anim2(frame):
    idx = frame * frame_step
    x1, x2 = imp_fine[idx, 0] - 1.5, imp_fine[idx, 1] + 1.5
    
    m1_s.set_data([x1], [0])
    m2_s.set_data([x2], [0])
    
    xl, yl = get_spring(-4, x1)
    xm, ym = get_spring(x1, x2, n_coils=5, width=0.15)
    xr, yr = get_spring(x2, 4)
    
    spr_left_s.set_data(xl, yl)
    spr_mid_s.set_data(xm, ym)
    spr_right_s.set_data(xr, yr)
    
    graph_rel.set_data(t_fine[:idx+1], imp_fine[:idx+1, 0] - imp_fine[:idx+1, 1])
    
    return m1_s, m2_s, spr_left_s, spr_mid_s, spr_right_s, graph_rel

ani2 = animation.FuncAnimation(fig_anim2, update_anim2, frames=frames_s, interval=50, blit=True)
HTML(ani2.to_jshtml())


## Выводы

### Зелёный уровень

При слабой связи $k_c=0.5$, $k_0=10$, $c=0$ наблюдаются биения, сопровождающиеся периодическим перераспределением энергии между массами.

Аналитические собственные частоты:

$$
\omega_1=\sqrt{10},\qquad \omega_2=\sqrt{11}.
$$

Аналитический период биений:

$$
T_{\mathrm{beat}}=\frac{2\pi}{\omega_2-\omega_1}
\approx40.708\text{ с}.
$$

В первоначальном расчёте численный период был определён неправильно, поскольку вместо огибающей анализировались максимумы быстрых колебаний.

После выделения огибающей численный период составил приблизительно $40.70$ с, что согласуется с аналитическим результатом.

RK4 и метод Верле были дополнительно исследованы на одинаковых шагах интегрирования.

При уменьшении шага погрешность RK4 убывает примерно как $O(h^4)$, а метода Верле — как $O(h^2)$.

При этом метод Верле демонстрирует ограниченные колебания полной механической энергии.

### Жёлтый уровень

Для исследования жёсткости выбраны параметры

$$
m=1,\quad k_0=2,\quad k_c=400,\quad c=10.
$$

В системе присутствуют медленная незатухающая синфазная мода и быстро затухающая относительная мода.

При шаге $h=0.12$ коэффициент усиления быстрой моды RK4 составляет

$$
|R_{\mathrm{RK4}}(h\lambda_{\mathrm{fast}})|
\approx2.658>1.
$$

Следовательно, численное решение RK4 становится неустойчивым. Прямое интегрирование подтверждает экспоненциальный рост амплитуды, несмотря на затухание соответствующей моды точного решения.

Неявный метод трапеций остаётся устойчивым и при увеличении шага, однако точность численного решения всё равно зависит от величины шага.

Проведён эксперимент с переключением шага с $0.01$ на $0.10$ после затухания быстрого переходного процесса.

Метод трапеций сохранил устойчивость и достаточно точно воспроизвёл медленное движение, в то время как RK4 продемонстрировал численный рост остаточной быстрой моды.

Таким образом, **после затухания быстрого процесса шаг интегрирования действительно можно увеличить при использовании A-устойчивого неявного метода**, но его величину необходимо выбирать с учётом требуемой точности медленной динамики.

Для RK4 ограничение устойчивости определяется также быстро затухающей модой и сохраняется даже после практически полного исчезновения соответствующего движения.
